# EDA - Buildings Master

Exploratory analysis of the buildings policies master file.

In [0]:
%pip install openpyxl seaborn

In [0]:
df_buildings = spark.read.parquet(
    "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/workings/policies_buildings_master.parquet"
)
display(df_buildings.limit(10))

In [0]:
# Trim to a smaller dataframe of features currently being used plus target and do EDA on that
# Year is derived in the next cell and appended there; period_start/period_end are kept
# in the select (not the variable list) because the Year cell derives from them.
variables_buildings = [
    "exposure", "policyholder_age", "sum_insured", "construction_type", "occupancy",
    "days_unoccupied", "smoke_alarms", "mortgage", "accidental_damage_cover", "voluntary_excess",
    "compulsory_excess", "channel", "payment_method", "missed_payments_12m", "new_business",
    "renewal_count", "days_quote_to_inception", "hail_days", "wind_exposure_index", "freezing_days",
    "dry_summer_index", "wildfire_index", "rainfall_mm", "mean_temperature_c", "occupation",
    "marital_status", "household_size", "credit_band", "prior_claims_5y", "licence_points",
    "years_at_address", "branch_id", "marketing_segment", "region", "urban_density", "flood_zone",
    "distance_to_river_km", "elevation_m", "soil_type", "year_built", "property_type",
    "roof_type", "basement", "trees_within_5m", "listed_building", "bedrooms", "energy_rating",
    "solar_panels", "ultimate_claims",
]

df_buildings_trim = df_buildings.select(
    "address_id",
    "customer_id",
    "row_id",
    "policy_id",
    "period_start",
    "period_end",
    "earned_premium",
    *variables_buildings,
).toPandas()
display(df_buildings_trim.head(10))

In [0]:
import pandas as pd

df_buildings_trim["Year"] = pd.to_datetime(df_buildings_trim["period_start"]).dt.year
variables_buildings.append("Year")

In [0]:
categorical_buildings = [
    c for c in variables_buildings
    if c not in ("period_start", "period_end")
    and (df_buildings_trim[c].dtype in ("object", "bool") or c == "Year")
]

counts = {c: df_buildings_trim[c].value_counts(dropna=False) for c in categorical_buildings}

for c, vc in counts.items():
    print(f"--- {c} ---")
    print(vc)

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Buildings_EDA.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    for c, vc in counts.items():
        vc.rename("count").rename_axis(c).to_excel(writer, sheet_name=c)

print(f"Saved value counts to {excel_path}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

numeric_buildings = [
    c for c in variables_buildings
    if c != "Year" and pd.api.types.is_numeric_dtype(df_buildings_trim[c])
]

for c in numeric_buildings:
    sns.boxplot(data=df_buildings_trim, y=c)
    plt.title(c)
    plt.show()

In [0]:
import io
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Buildings_EDA.xlsx"

numeric_buildings = [
    c for c in variables_buildings
    if c != "Year" and pd.api.types.is_numeric_dtype(df_buildings_trim[c])
]

wb = load_workbook(excel_path)

for c in numeric_buildings:
    fig, ax = plt.subplots()
    sns.boxplot(data=df_buildings_trim, y=c, ax=ax)
    ax.set_title(c)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    plt.close(fig)
    buf.seek(0)
    img = XLImage(buf)
    ws = wb.create_sheet(f"{c}_boxplot"[:31])
    ws.add_image(img, "B2")

wb.save(excel_path)
print(f"Added {len(numeric_buildings)} boxplots to {excel_path}")

In [0]:
over_one = df_buildings_trim[df_buildings_trim["exposure"] > 1]

print(f"Rows with exposure > 1: {len(over_one):,} of {len(df_buildings_trim):,} ({len(over_one) / len(df_buildings_trim):.2%})")
print(over_one["exposure"].describe())

excel_path = "/Workspace/Users/bridget.macdonnell1@gmail.com/hackathon-project/Excel files/Buildings_EDA.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    over_one.to_excel(writer, sheet_name="exposure_over_1", index=False)

print(f"Saved {len(over_one):,} rows to {excel_path} (sheet: exposure_over_1)")